In [0]:
# Importações
from pyspark.sql import functions as F

# Bronze
bronze_acidentes = spark.table("workspace.bronze.acidentes")
bronze_localidade = spark.table("workspace.bronze.localidade")
bronze_tipo_veiculo = spark.table("workspace.bronze.tipo_veiculo")
bronze_vitimas = spark.table("workspace.bronze.vitimas")

# Silver
silver_acidentes = spark.table("workspace.silver.acidentes")
silver_localidade = spark.table("workspace.silver.localidade")
silver_tipo_veiculo = spark.table("workspace.silver.tipo_veiculo")
silver_vitimas = spark.table("workspace.silver.vitimas")

# Gold
gold_dim_tempo = spark.table("workspace.gold.dim_tempo")
gold_dim_horario = spark.table("workspace.gold.dim_horario")
gold_dim_bairro = spark.table("workspace.gold.dim_bairro")
gold_dim_regiao = spark.table("workspace.gold.dim_regiao")
gold_fato_acidentes = spark.table("workspace.gold.fato_acidentes")

print("Tabelas Bronze, Silver e Gold carregadas com sucesso.")

Tabelas Bronze, Silver e Gold carregadas com sucesso.


In [0]:
# Avaliação de completude dos principais atributos da Silver

colunas_completude = [
    "num_acidente",
    "data_acidente",
    "hora_acidente",
    "bairro_acidente",
    "tp_acidente",
    "cond_meteorologica",
    "cond_pista",
    "fase_dia",
    "qtde_envolvidos",
    "qtde_feridosilesos",
    "qtde_obitos"
]

total_acidentes = silver_acidentes.count()

resultado_completude = []

for coluna in colunas_completude:
    qtd_nulos = (
        silver_acidentes
        .filter(F.col(coluna).isNull())
        .count()
    )

    qtd_preenchidos = total_acidentes - qtd_nulos

    percentual_completude = (
        qtd_preenchidos / total_acidentes * 100
    )

    resultado_completude.append(
        (
            coluna,
            total_acidentes,
            qtd_preenchidos,
            qtd_nulos,
            round(percentual_completude, 2)
        )
    )

df_completude = spark.createDataFrame(
    resultado_completude,
    [
        "atributo",
        "total_registros",
        "preenchidos",
        "nulos",
        "completude_percentual"
    ]
)

display(
    df_completude
    .orderBy("completude_percentual")
)

atributo,total_registros,preenchidos,nulos,completude_percentual
bairro_acidente,55046,28152,26894,51.14
num_acidente,55046,55046,0,100.0
data_acidente,55046,55046,0,100.0
hora_acidente,55046,55046,0,100.0
tp_acidente,55046,55046,0,100.0
cond_meteorologica,55046,55046,0,100.0
cond_pista,55046,55046,0,100.0
fase_dia,55046,55046,0,100.0
qtde_envolvidos,55046,55046,0,100.0
qtde_feridosilesos,55046,55046,0,100.0


In [0]:
# Avaliação temporal da completude do atributo bairro

qualidade_bairro_ano = (
    silver_acidentes
    .groupBy("ano_acidente")
    .agg(
        F.count("*").alias("total_acidentes"),

        F.sum(
            F.col("bairro_acidente").isNotNull().cast("int")
        ).alias("bairro_preenchido"),

        F.sum(
            F.col("bairro_acidente").isNull().cast("int")
        ).alias("bairro_nulo")
    )
    .withColumn(
        "completude_bairro_percentual",
        F.round(
            F.col("bairro_preenchido")
            / F.col("total_acidentes") * 100,
            2
        )
    )
    .orderBy("ano_acidente")
)

display(qualidade_bairro_ano)

ano_acidente,total_acidentes,bairro_preenchido,bairro_nulo,completude_bairro_percentual
2018,9683,0,9683,0.0
2019,9449,1,9448,0.01
2020,6036,15,6021,0.25
2021,6918,6511,407,94.12
2022,7619,6922,697,90.85
2023,7799,7649,150,98.08
2024,7542,7054,488,93.53


In [0]:
# Avaliação de unicidade do identificador do acidente

total_registros = silver_acidentes.count()

total_identificadores = (
    silver_acidentes
    .select("num_acidente")
    .distinct()
    .count()
)

identificadores_duplicados = (
    silver_acidentes
    .groupBy("num_acidente")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

identificadores_nulos = (
    silver_acidentes
    .filter(F.col("num_acidente").isNull())
    .count()
)

print(f"Total de registros: {total_registros}")
print(f"Identificadores distintos: {total_identificadores}")
print(f"Identificadores duplicados: {identificadores_duplicados}")
print(f"Identificadores nulos: {identificadores_nulos}")

if (
    total_registros == total_identificadores
    and identificadores_duplicados == 0
    and identificadores_nulos == 0
):
    print("Resultado: num_acidente atende ao critério de unicidade.")
else:
    print("Resultado: foram identificados problemas de unicidade.")

Total de registros: 55046
Identificadores distintos: 55046
Identificadores duplicados: 0
Identificadores nulos: 0
Resultado: num_acidente atende ao critério de unicidade.


In [0]:
# Avaliação de validade e consistência dos atributos temporais

datas_nulas = (
    silver_acidentes
    .filter(F.col("data_acidente").isNull())
    .count()
)

horarios_nulos = (
    silver_acidentes
    .filter(F.col("hora_acidente").isNull())
    .count()
)

# Ano armazenado deve ser coerente com a data
ano_inconsistente = (
    silver_acidentes
    .filter(
        F.year("data_acidente")
        != F.col("ano_acidente").cast("int")
    )
    .count()
)

# Mês armazenado deve ser coerente com a data
mes_inconsistente = (
    silver_acidentes
    .filter(
        F.month("data_acidente")
        != F.col("mes_acidente").cast("int")
    )
    .count()
)

# Limites observados
limites_temporais = (
    silver_acidentes
    .agg(
        F.min("data_acidente").alias("data_minima"),
        F.max("data_acidente").alias("data_maxima"),
        F.min(F.hour("hora_acidente")).alias("hora_minima"),
        F.max(F.hour("hora_acidente")).alias("hora_maxima")
    )
)

print(f"Datas nulas: {datas_nulas}")
print(f"Horários nulos: {horarios_nulos}")
print(f"Ano inconsistente com data: {ano_inconsistente}")
print(f"Mês inconsistente com data: {mes_inconsistente}")

print("\nLimites temporais observados:")
display(limites_temporais)

Datas nulas: 0
Horários nulos: 0
Ano inconsistente com data: 0
Mês inconsistente com data: 0

Limites temporais observados:


data_minima,data_maxima,hora_minima,hora_maxima
2018-01-01,2024-11-30,0,23


In [0]:
# Avaliação das principais medidas quantitativas

medidas = [
    "qtde_acidente",
    "qtde_envolvidos",
    "qtde_feridosilesos",
    "qtde_obitos"
]

resultado_medidas = []

for medida in medidas:
    resultado = (
        silver_acidentes
        .agg(
            F.sum(F.col(medida).isNull().cast("int")).alias("nulos"),
            F.sum((F.col(medida) < 0).cast("int")).alias("negativos"),
            F.min(medida).alias("minimo"),
            F.max(medida).alias("maximo")
        )
        .first()
    )

    resultado_medidas.append(
        (
            medida,
            resultado["nulos"],
            resultado["negativos"],
            resultado["minimo"],
            resultado["maximo"]
        )
    )

df_validade_medidas = spark.createDataFrame(
    resultado_medidas,
    ["medida", "nulos", "negativos", "minimo", "maximo"]
)

display(df_validade_medidas)

# Consistências relacionais básicas
obitos_maior_envolvidos = (
    silver_acidentes
    .filter(F.col("qtde_obitos") > F.col("qtde_envolvidos"))
    .count()
)

feridosilesos_maior_envolvidos = (
    silver_acidentes
    .filter(F.col("qtde_feridosilesos") > F.col("qtde_envolvidos"))
    .count()
)

soma_maior_envolvidos = (
    silver_acidentes
    .filter(
        (F.col("qtde_obitos") + F.col("qtde_feridosilesos"))
        > F.col("qtde_envolvidos")
    )
    .count()
)

print(f"Óbitos > envolvidos: {obitos_maior_envolvidos}")
print(f"Feridos/ilesos > envolvidos: {feridosilesos_maior_envolvidos}")
print(f"Óbitos + feridos/ilesos > envolvidos: {soma_maior_envolvidos}")

medida,nulos,negativos,minimo,maximo
qtde_acidente,0,0,1,1
qtde_envolvidos,0,0,0,28
qtde_feridosilesos,0,0,0,28
qtde_obitos,0,0,0,4


Óbitos > envolvidos: 0
Feridos/ilesos > envolvidos: 0
Óbitos + feridos/ilesos > envolvidos: 0


In [0]:
# Avaliação da integridade referencial da tabela fato

dimensoes = [
    ("id_tempo", gold_dim_tempo),
    ("id_horario", gold_dim_horario),
    ("id_bairro", gold_dim_bairro),
    ("id_regiao", gold_dim_regiao)
]

resultado_integridade = []

for chave, dimensao in dimensoes:

    chaves_nulas = (
        gold_fato_acidentes
        .filter(F.col(chave).isNull())
        .count()
    )

    chaves_orfas = (
        gold_fato_acidentes
        .select(chave)
        .distinct()
        .join(
            dimensao.select(chave),
            chave,
            "left_anti"
        )
        .count()
    )

    resultado_integridade.append(
        (
            chave,
            chaves_nulas,
            chaves_orfas
        )
    )

df_integridade = spark.createDataFrame(
    resultado_integridade,
    [
        "chave",
        "valores_nulos",
        "chaves_orfas"
    ]
)

display(df_integridade)

chave,valores_nulos,chaves_orfas
id_tempo,0,0
id_horario,0,0
id_bairro,0,0
id_regiao,0,0


In [0]:
# Fonte oficial de bairros e Regiões Administrativas
# Instituto Pereira Passos / Prefeitura da Cidade do Rio de Janeiro

import requests

url_bairros_ipp = (
    "https://pgeo3.rio.rj.gov.br/arcgis/rest/services/"
    "Cartografia/Limites_administrativos/FeatureServer/4/query"
)

params = {
    "where": "1=1",
    "outFields": "nome,codbairro,regiao_adm,codra,area_plane",
    "returnGeometry": "false",
    "f": "json"
}

response = requests.get(url_bairros_ipp, params=params, timeout=60)
response.raise_for_status()

dados_ipp = response.json()

if "error" in dados_ipp:
    raise Exception(dados_ipp["error"])

df_bairros_ipp_quality = spark.createDataFrame(
    [feature["attributes"] for feature in dados_ipp["features"]]
)

# Normalização determinística apenas para comparação
bairros_ipp_quality = (
    df_bairros_ipp_quality
    .withColumn(
        "bairro_match",
        F.upper(
            F.translate(
                F.trim(F.col("nome")),
                "ÁÀÃÂÉÊÍÓÔÕÚÜÇáàãâéêíóôõúüç",
                "AAAAEEIOOOUUCaaaaeeiooouuc"
            )
        )
    )
    .select("bairro_match")
    .distinct()
)

acidentes_quality = (
    silver_acidentes
    .withColumn(
        "bairro_match",
        F.upper(
            F.translate(
                F.trim(F.col("bairro_acidente")),
                "ÁÀÃÂÉÊÍÓÔÕÚÜÇáàãâéêíóôõúüç",
                "AAAAEEIOOOUUCaaaaeeiooouuc"
            )
        )
    )
)

resultado_territorial = (
    acidentes_quality.alias("a")
    .join(
        bairros_ipp_quality.alias("i"),
        F.col("a.bairro_match") == F.col("i.bairro_match"),
        "left"
    )
    .withColumn(
        "status_territorial",
        F.when(
            F.col("a.bairro_acidente").isNull(),
            "SEM_BAIRRO"
        )
        .when(
            F.col("i.bairro_match").isNotNull(),
            "BAIRRO_OFICIAL_ASSOCIADO"
        )
        .otherwise(
            "BAIRRO_NAO_ASSOCIADO"
        )
    )
    .groupBy("status_territorial")
    .count()
    .withColumn(
        "percentual",
        F.round(
            F.col("count") / F.lit(55046) * 100,
            2
        )
    )
    .orderBy(F.desc("count"))
)

display(resultado_territorial)

status_territorial,count,percentual
BAIRRO_OFICIAL_ASSOCIADO,27075,49.19
SEM_BAIRRO,26894,48.86
BAIRRO_NAO_ASSOCIADO,1077,1.96


In [0]:
# Reconciliação da quantidade de acidentes entre as camadas

# Bronze: recorte correspondente ao município do Rio de Janeiro
bronze_rio = (
    bronze_acidentes
    .filter(F.col("codigo_ibge") == "3304557")
    .count()
)

# Silver: acidentes tratados do município do Rio de Janeiro
silver_rio = silver_acidentes.count()

# Gold: granularidade da fato = 1 registro por acidente
gold_rio = gold_fato_acidentes.count()

diferenca_bronze_silver = bronze_rio - silver_rio
diferenca_silver_gold = silver_rio - gold_rio

print(f"Bronze - Rio de Janeiro: {bronze_rio}")
print(f"Silver - Rio de Janeiro: {silver_rio}")
print(f"Gold - fato_acidentes: {gold_rio}")

print(f"\nDiferença Bronze → Silver: {diferenca_bronze_silver}")
print(f"Diferença Silver → Gold: {diferenca_silver_gold}")

if (
    diferenca_bronze_silver == 0
    and diferenca_silver_gold == 0
):
    print(
        "\nResultado: quantidade de acidentes preservada "
        "entre Bronze, Silver e Gold."
    )
else:
    print(
        "\nResultado: existe diferença quantitativa entre as camadas."
    )

Bronze - Rio de Janeiro: 55046
Silver - Rio de Janeiro: 55046
Gold - fato_acidentes: 55046

Diferença Bronze → Silver: 0
Diferença Silver → Gold: 0

Resultado: quantidade de acidentes preservada entre Bronze, Silver e Gold.


In [0]:
# Resumo consolidado dos principais resultados de qualidade

resumo_qualidade = [
    (
        "Completude",
        "bairro_acidente",
        "51,14% preenchido",
        "ATENCAO",
        "Ausência concentrada principalmente entre 2018 e 2020."
    ),
    (
        "Completude",
        "Demais atributos críticos avaliados",
        "100% preenchidos",
        "OK",
        "Sem valores nulos após tratamento Silver."
    ),
    (
        "Unicidade",
        "num_acidente",
        "0 duplicados",
        "OK",
        "55.046 identificadores distintos para 55.046 registros."
    ),
    (
        "Validade temporal",
        "data_acidente / hora_acidente",
        "0 inconsistências",
        "OK",
        "Datas entre 2018-01-01 e 2024-11-30; horas entre 0 e 23."
    ),
    (
        "Validade quantitativa",
        "Medidas do acidente",
        "0 violações",
        "OK",
        "Sem negativos e sem violações das relações quantitativas testadas."
    ),
    (
        "Integridade referencial",
        "Gold",
        "0 chaves órfãs",
        "OK",
        "As quatro FKs da fato possuem correspondência nas dimensões."
    ),
    (
        "Consistência territorial",
        "Bairro x IPP",
        "49,19% associados",
        "ATENCAO",
        "48,86% sem bairro e 1,96% com bairro não associado à referência oficial."
    ),
    (
        "Reconciliação",
        "Bronze -> Silver -> Gold",
        "55.046 -> 55.046 -> 55.046",
        "OK",
        "Nenhuma perda ou multiplicação de acidentes entre as camadas."
    )
]

df_resumo_qualidade = spark.createDataFrame(
    resumo_qualidade,
    [
        "dimensao_qualidade",
        "objeto_avaliado",
        "resultado",
        "status",
        "observacao"
    ]
)

display(df_resumo_qualidade)

dimensao_qualidade,objeto_avaliado,resultado,status,observacao
Completude,bairro_acidente,"51,14% preenchido",ATENCAO,Ausência concentrada principalmente entre 2018 e 2020.
Completude,Demais atributos críticos avaliados,100% preenchidos,OK,Sem valores nulos após tratamento Silver.
Unicidade,num_acidente,0 duplicados,OK,55.046 identificadores distintos para 55.046 registros.
Validade temporal,data_acidente / hora_acidente,0 inconsistências,OK,Datas entre 2018-01-01 e 2024-11-30; horas entre 0 e 23.
Validade quantitativa,Medidas do acidente,0 violações,OK,Sem negativos e sem violações das relações quantitativas testadas.
Integridade referencial,Gold,0 chaves órfãs,OK,As quatro FKs da fato possuem correspondência nas dimensões.
Consistência territorial,Bairro x IPP,"49,19% associados",ATENCAO,"48,86% sem bairro e 1,96% com bairro não associado à referência oficial."
Reconciliação,Bronze -> Silver -> Gold,55.046 -> 55.046 -> 55.046,OK,Nenhuma perda ou multiplicação de acidentes entre as camadas.
